---
## 1. Install dependencies

In [0]:
# Install kagglehub with pandas support; %restart_python ensures the new package is available in this session
%pip install kagglehub[pandas-datasets]
#%restart_python

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


---
## 2. Download the dataset from Kaggle

Dataset link 1: [Open Food Facts](https://www.kaggle.com/datasets/alexandrelemercier/food-detailed-nutritional-content) - Kaggle

Dataset link 2: [Food Ingredient List](https://www.kaggle.com/datasets/datafiniti/food-ingredient-lists) - Kaggle


In [0]:
#Dataset 1
# Import kagglehub and the adapter that returns Pandas DataFrames
import kagglehub
from kagglehub import KaggleDatasetAdapter

# Which file from the dataset to load (this dataset has ultrafeedback.csv)
file_path = "openfoodfacts_cleaned.csv"

# Load the latest version of the dataset as a Pandas DataFrame.
# Dataset: "thedrcat/llm-human-preference-data-ultrafeedback"
# For more options (e.g. sql_query, pandas_kwargs), see:
# https://github.com/Kaggle/kagglehub/blob/main/README.md#kaggledatasetadapterpandas
df = kagglehub.dataset_load(
    KaggleDatasetAdapter.PANDAS,
    "alexandrelemercier/food-detailed-nutritional-content",
    file_path,
)

print("First 5 records:", df.head())
print("\nShape:", df.shape)

100%|██████████| 92.8M/92.8M [00:01<00:00, 55.1MB/s]

Extracting zip of openfoodfacts_cleaned.csv...



/local_disk0/.ephemeral_nfs/envs/pythonEnv-625faf4f-a892-4738-9784-a03748dc858f/lib/python3.12/site-packages/kagglehub/pandas_datasets.py:92: DtypeWarning: Columns (6) have mixed types. Specify dtype option on import or set low_memory=False.
  result = read_function(


First 5 records:    Unnamed: 0  ... nutrition-score-fr_100g
0           3  ...                    -5.0
1           4  ...                     NaN
2           8  ...                     NaN
3           9  ...                     NaN
4          12  ...                     NaN

[5 rows x 36 columns]

Shape: (1982824, 36)


In [0]:
import re

# Create a copy to preserve original
df_cleaned = df.copy()

# 1. Drop the 'Unnamed: 0' column (causes Delta save issues)
if 'Unnamed: 0' in df_cleaned.columns:
    df_cleaned = df_cleaned.drop(columns=['Unnamed: 0'])
    print("✓ Removed 'Unnamed: 0' column")

# 2. Fix unicode encoding issues in text columns
text_columns = ['product_name', 'generic_name', 'brands', 'countries_en', 'categories_en', 'origins_en', 'traces_en', 'additives_en']

# Common unicode replacements for incorrectly encoded characters
unicode_fixes = {
    'Ã©': 'é', 'Ã¨': 'è', 'Ã': 'à', 'Ã´': 'ô', 'Ã®': 'î', 'Ã»': 'û',
    'Ã§': 'ç', 'Ã¢': 'â', 'Ã«': 'ë', 'Ã¯': 'ï', 'Ã¼': 'ü', 'Ã¶': 'ö',
    'Ã±': 'ñ', 'Ã': 'Ã', 'â': "'", 'Â': '', 'â¢': '•',
    'râpées': 'râpées', 'Pâte': 'Pâte', 'pâte': 'pâte'
}

for col in text_columns:
    if col in df_cleaned.columns:
        # Replace common unicode issues
        for bad, good in unicode_fixes.items():
            df_cleaned[col] = df_cleaned[col].str.replace(bad, good, regex=False)

print("✓ Fixed unicode encoding issues in text columns")

# 3. Standardize country names to include 'United States' (handle variations)
if 'countries_en' in df_cleaned.columns:
    # Replace common variations with 'United States'
    country_replacements = {
        r'\bUSA\b': 'United States',
        r'\bUS\b': 'United States',
        r'\bU\.S\.\b': 'United States',
        r'\bU\.S\.A\.\b': 'United States',
        r'\bUnited States of America\b': 'United States'
    }
    
    for pattern, replacement in country_replacements.items():
        df_cleaned['countries_en'] = df_cleaned['countries_en'].str.replace(
            pattern, replacement, regex=True, case=False
        )
    
    print("✓ Standardized country names")

# 4. Filter to keep only rows that contain 'United States'
if 'countries_en' in df_cleaned.columns:
    us_mask = df_cleaned['countries_en'].astype(str).str.contains('United States', case=False, na=False)
    df_cleaned = df_cleaned[us_mask].reset_index(drop=True)
    print(f"✓ Filtered to rows containing 'United States'")

# 5. Show summary of changes
print(f"\nCleaned DataFrame shape: {df_cleaned.shape}")
print(f"Original DataFrame shape: {df.shape}")
print(f"\nTop 10 countries after cleaning:")
print(df_cleaned['countries_en'].value_counts().head(10))

# Show before/after examples of unicode fixes
print("\n" + "="*80)
print("Unicode fix examples:")
sample_indices = df[df['product_name'].astype(str).str.contains('Ã', na=False)].head(3).index
for idx in sample_indices:
    if idx in df.index:
        print(f"Before: {df.loc[idx, 'product_name']}")
        if idx in df_cleaned.index:
            print(f"After:  {df_cleaned.loc[idx, 'product_name']}")
        print()

✓ Removed 'Unnamed: 0' column
✓ Fixed unicode encoding issues in text columns
✓ Standardized country names
✓ Filtered to rows containing 'United States'

Cleaned DataFrame shape: (472957, 35)
Original DataFrame shape: (1982824, 36)

Top 10 countries after cleaning:
countries_en
United States                   464902
France,United States              3647
Germany,United States             1038
Canada,United States               336
Spain,United States                303
Bolivia,United States              253
Mexico,United States               224
United Kingdom,United States       192
France,Germany,United States       114
Australia,United States            111
Name: count, dtype: int64

Unicode fix examples:
Before: Pã¢te De Piment Ã Avec Huile De Soja 454G
After:  Almond cherry flavored plant-based protein bar, almond cherry

Before: Foie gras de canard entier mi cuit aromatisÃ© au JuranÃ§on

Before: Chips Ã Cuire Goã»t Crevettes 1 KG



In [0]:
# Unity Catalog location: catalog.schema.table
catalog = "main"
schema = "default"
table_name = "open_food_facts"

# Convert cleaned Pandas DataFrame to Spark and write as Delta
# mode("overwrite") replaces the table if it exists (good for re-running the lab)
open_food_facts = spark.createDataFrame(df_cleaned)
open_food_facts.write.format("delta").mode("overwrite").saveAsTable(f"{catalog}.{schema}.{table_name}")

# Verify: query the table (optional)
display(spark.table(f"{catalog}.{schema}.{table_name}").limit(5))

product_name,generic_name,quantity,brands,categories_en,origins_en,countries_en,traces_en,additives_n,additives_en,nutriscore_score,food_groups_en,ecoscore_score,ecoscore_grade,main_category_en,energy-kcal_100g,fat_100g,saturated-fat_100g,monounsaturated-fat_100g,polyunsaturated-fat_100g,trans-fat_100g,cholesterol_100g,carbohydrates_100g,sugars_100g,fiber_100g,proteins_100g,salt_100g,sodium_100g,vitamin-a_100g,vitamin-c_100g,potassium_100g,calcium_100g,iron_100g,fruits-vegetables-nuts-estimate-from-ingredients_100g,nutrition-score-fr_100g
Mini fudge stripes dark chocolate cookies,null,null,null,"Snacks,Sweet snacks,Biscuits and cakes,Biscuits",null,United States,null,5.0,"E322 - Lecithins,E322i - Lecithin,E435 - Polyoxyethylene sorbitan monostearate,E472e - Mono- and diacetyltartaric acid esters of mono- and diglycerides of fatty acids,E491 - Sorbitan monostearate,E500 - Sodium carbonates,E500ii - Sodium hydrogen carbonate",22.0,"Sugary snacks,Biscuits and cakes",39.0,d,Biscuits,476.0,19.05,14.29,2.38,2.38,0.0,0.0,71.43,38.1,4.8,4.76,0.9525,0.381,0.0,0.0,null,0.0,0.00171,0.855580357142857,22.0
"Coconut chips deluxe cookies, coconut",null,null,null,"Snacks,Sweet snacks,Biscuits and cakes,Biscuits",null,United States,null,0.0,null,22.0,"Sugary snacks,Biscuits and cakes",49.0,c,Biscuits,533.0,30.0,13.33,null,null,0.0,0.0,60.0,30.0,3.3,6.67,0.75,0.3,0.0,0.0,null,0.0,0.0024,7.5,22.0
"Pecan shortbread cookies, pecan",null,null,null,"Snacks,Sweet snacks,Biscuits and cakes,Biscuits",null,United States,null,2.0,"E322 - Lecithins,E322i - Lecithin,E500 - Sodium carbonates,E500ii - Sodium hydrogen carbonate",20.0,"Sugary snacks,Biscuits and cakes",49.0,c,Biscuits,548.0,32.26,9.68,null,null,0.0,0.016,58.06,22.58,3.2,6.45,0.8875,0.355,0.0,0.0,null,0.0,0.00232,5.55555555555556,20.0
"Keebler, chips deluxe cookies, cookies, chocolate lovers",null,null,null,"Snacks,Sweet snacks,Biscuits and cakes,Biscuits",null,United States,null,2.0,"E322 - Lecithins,E322i - Lecithin,E500 - Sodium carbonates,E500ii - Sodium hydrogen carbonate",24.0,"Sugary snacks,Biscuits and cakes",39.0,d,Biscuits,531.0,28.12,12.5,null,null,0.0,0.016,62.5,34.38,3.1,6.25,0.9375,0.375,0.0,0.0,null,0.0,0.00338,11.5384615384615,24.0
"Simply shortbread cookies, simply",null,null,null,"Snacks,Sweet snacks,Biscuits and cakes,Biscuits",null,United States,null,1.0,"E500 - Sodium carbonates,E500ii - Sodium hydrogen carbonate",24.0,"Sugary snacks,Biscuits and cakes",49.0,c,Biscuits,516.0,29.03,12.9,null,null,0.0,0.048,61.29,22.58,0.0,6.45,0.725,0.29,9.69E-5,0.0,null,0.0,0.00232,0.0,24.0


In [0]:
#Dataset 2
import pandas as pd
import kagglehub
import os

# Download dataset and get path
path = kagglehub.dataset_download("datafiniti/food-ingredient-lists")

# Load the CSV file with robust parsing options
df2 = pd.read_csv(
    os.path.join(path, 'ingredients v1.csv'),
    encoding='latin-1',
    on_bad_lines='skip',
    engine='python'
)

print("First 5 records:")
print(df2.head())
print("\nShape:", df2.shape)

First 5 records:
                  ï»¿id                  asins  ...       weight Unnamed: 15
0  AVphBRHOilAPnD_x0OrE             B00HXST15C  ...  10.6 pounds         NaN
1  AVpfNFy1LJeJML434ma2  B008VT0W8C,B0092F8OJ8  ...   3.5 ounces         NaN
2  AVpgT49VLJeJML43MJEz             B00CHTWZ2S  ...   1.8 pounds         NaN
3  AVphYgnzLJeJML43aPp2             B002JJYNVW  ...   1.6 pounds         NaN
4  AVpiS0bOLJeJML43kRsh             B00290W1CY  ...    18 pounds         NaN

[5 rows x 16 columns]

Shape: (10000, 16)


In [0]:
df2.columns

Index(['ï»¿id', 'asins', 'brand', 'categories', 'dateAdded', 'dateUpdated',
       'ean', 'features.key', 'features.value', 'manufacturer',
       'manufacturerNumber', 'name', 'sizes', 'upc', 'weight', 'Unnamed: 15'],
      dtype='object')

In [0]:
# Unity Catalog location: catalog.schema.table
catalog = "main"
schema = "default"
table_name = "food_ingredient_lists"

# Clean column names to remove invalid characters for Delta
df2_cleaned = df2.rename(columns=lambda x: x.replace('ï»¿', '').replace(':', '_').replace(' ', '_'))

# Convert Pandas DataFrame to Spark and write as Delta
# mode("overwrite") replaces the table if it exists (good for re-running the lab)
food_ingredient_lists = spark.createDataFrame(df2_cleaned)
food_ingredient_lists.write.format("delta").mode("overwrite").saveAsTable(f"{catalog}.{schema}.{table_name}")

# Verify: query the table (optional)
display(spark.table(f"{catalog}.{schema}.{table_name}").limit(5))

id,asins,brand,categories,dateAdded,dateUpdated,ean,features.key,features.value,manufacturer,manufacturerNumber,name,sizes,upc,weight,Unnamed__15
AVpiV3Fs1cnluZ0-MVMl,B004DED9LQ,Thank You Tins,"Food,Food Gifts,Grocery & Gourmet Food,Candy & Chocolate,Fudge",2015-10-30T20:45:48Z,2017-09-12T19:59:25Z,null,Ingredients,"Sugar,glucose syrup,sweetened condensed skim milk,palm oil,butter,vanilla flavoring,salt",null,9098,Vanilla Fudge Thank You Tins - Languages,null,7.04039E+11,14.1 ounces,null
AVpfiO841cnluZ0-meXR,"B00MQ6FLH8,B01LZGV8M3,B00HT88UNA",Jack Links,"Jerky & Dried Meats,Grocery & Gourmet Food,Snack Foods,Food,Snacks, Cookies & Chips,Jerky,Food & Grocery,General Grocery,Jerky & Meat Snacks,grocery",2015-05-11T13:04:17Z,2017-09-06T17:28:19Z,17082001382,Ingredients,"Beef Stick: Beef, Salt, Corn Syrup, Spices, contains 2% or less of Water, Dextrose, Sugar, Flavorings, Sodium Erythorbate, Lactic Acid Starter Culture, Sodium Nitrite, BHA, BHT, Citric Acid, treated with a solution of Potassium Sorbate to ensure freshness.Pasteurized Process Cheddar Cheese Food: Cheddar Cheese (Cultured Milk, Salt, Enzymes), Water, Sodium Phosphate, Natural Flavoring, Salt, Sorbic Acid (preservative), Color (Paprika and Turmeric Extract).,Beef Stick: Beef,Salt,Corn Syrup,Spices,contains 2% or less of Water,Dextrose,Sugar,Flavorings,Sodium Erythorbate,Lactic Acid Starter Culture,Sodium Nitrite,BHA,BHT,Citric Acid,treated with a solution of Potassium Sorbate to ensure freshness.Pasteurized Process Cheddar Cheese Food: Cheddar Cheese (Cultured Milk,Enzymes),Water,Sodium Phosphate,Natural Flavoring,Sorbic Acid (preservative),Color (Paprika and Turmeric Extract).",Jack Links,1038,"Jack Links All-american Beef & Cheese Combo, 7.2 Ounce",null,17082001382,12.2 ounces,null
AV1guauIGV-KLJ3ad334,B006GF1J6C,Whitman's,"Food,Food Gifts,Grocery & Gourmet Food,Candy & Chocolate,Assortments",2017-07-20T15:59:27Z,2017-09-12T19:59:24Z,null,Ingredients,"Ingredients consist of Chocolate Candy {Maltitol,Chocolate,Cocoa Butter,Sodium Caseinate (Milk),Milk Fat,Soy Lecithin (an Emulsifier),Sucralose,Natural and Artificial Flavor,Salt},Maltitol Syrup,Dark Chocolate {Maltitol,Chocolate (processed with Alkali),Milk Fat (Milk),Salt,Vanillin (an Artificial Flavor),Sucralose},Peanut Butter (Dry Roasted Peanuts,Hydrogenated Cottonseed/Rapeseed Oil,Soy Oil,Salt),Polydextrose,Palm Kernel Oil,Butter,Pastel Coating {Maltitol,Fractionated Palm Kernel Oil and Hydrogenated Palm Oil,Lactic Acid Esters of Mono- and Diglycerides with Citric Acid (to help protect Flavor),Natural Flavor,Peanuts,Evaporated Milk (Milk,Dipotassium Phosphate,Carrageenan,Vitamin D),Maltitol,Coconut with Sodium Metabisulfite (to promote Color Retention),Sorbitol,Walnuts,Almonds,Polyglycerol (Hydrogenated Starch Hydrolysate),Hydrogenated Vegetable Oils (Palm Kernel,Soybean),Natural and Artificial Flavors*,Sodium Benzoate and Potassium Sorbate (preservatives),Mono and Diglycerides with Citric Acid (to help protect Flavor),Orange Oil,Spice,Tocopherols (a Natural Antioxidant),Dairy Cream,and FD&C Colors (Yellow #5 & #6,Red #40 Blue #1).*Adds a negligible amount of Sugar.,Chocolate Candy [Maltitol, Chocolate, Cocoa Butter, Sodium Caseinate (Milk), Milk Fat, Soy Lecithin/an Emulsifier, Sucralose, Natural and Artificial Flavor, and Salt], Maltitol Syrup, Dark Chocolate (Maltitol, Unsweetened Chocolate, Cocoa Butter, Milk Fat, Soy Lecithin/an Emulsifier, Salt, Artificial Flavour, and Sucralose), Partially Hydrogenated Vegetable Oils (Palm Kernel, Soybean, Cottonseed), Polydextrose, Peanut Butter [Peanuts, Hydrogenated Vegetable Oils (Rapeseed, Cottonseed, Soybean), and Salt], Butter, Nuts (Peanuts, Walnuts, Almonds), Pastel Coating [Maltitol, Fractionated Palm Kernel Oil and Partially Hydrogenated Palm Oil, Sodium Caseinate (Milk), Milk Fat, Glycerol-Lacto Esters of Fatty Acids with Citric Acid, Soy Lecithin/an Emulsifier, Vanillin/an Artificial Flavor, Natural Flavor, and Sucralose), Maltitol, Evaporated Milk (Milk, Dip

In [0]:
import pandas as pd
import os

# Get the directory path where the notebook is running
current_dir = os.getcwd()


# Read the FDA_Fodd_Additives.csv file into a DataFrame
df = pd.read_csv(os.path.join(current_dir, "FDA_Food_Additives.csv"))

# Optionally show the first few records
print("First 5 records:")
print(df.head())
print("\nShape:", df.shape)

First 5 records:
                              Substance  ...  JECFA Flavor Number
0                    ACAI BERRY EXTRACT  ...                  NaN
1                  ACESULFAME POTASSIUM  ...                  NaN
2                                ACETAL  ...                  941
3                          ACETALDEHYDE  ...                   80
4  ACETALDEHYDE, BUTYL PHENETHYL ACETAL  ...                 1001

[5 rows x 18 columns]

Shape: (3971, 18)


In [0]:
# Unity Catalog location: catalog.schema.table
catalog = "main"
schema = "default"
table_name = "fda_food_additives"

# Clean column names to remove invalid characters for Delta
df = df.rename(columns=lambda x: x.replace('ï»¿', '').replace(':', '_').replace(' ', '_').replace('(', '_').replace(')', '_'))

# Convert Pandas DataFrame to Spark and write as Delta
# mode("overwrite") replaces the table if it exists (good for re-running the lab)
fda_food_additives = spark.createDataFrame(df)
fda_food_additives.write.format("delta").mode("overwrite").saveAsTable(f"{catalog}.{schema}.{table_name}")

# Verify: query the table (optional)
display(spark.table(f"{catalog}.{schema}.{table_name}").limit(5))

Substance,Health_Risk_Score_1_to_10,Health_Risk_Level,Synthetic_Content_Score_1_to_10,Clean_Label_Concern_Score_1_to_10,Ultra_Processed_Association,Risk_Categories,Functional_Category,Technical_Function,Suggested_Alternatives,Allergen_or_Sensitivity_Risk,Common_Food_Uses,Regulatory_Status,CAS_Number,Regulatory_References__21_CFR_,FEMA_GRAS_Number,GRAS_Publication_Number,JECFA_Flavor_Number
"DOG GRASS, EXTRACT (AGROPYRON REPENS (L.) BEAUV.)",3,Safe,2,1,No,General_Approved,Flavoring Agent,"FLAVOR ENHANCER, FLAVORING AGENT OR ADJUVANT","Natural plant extracts, essential oils, or fermentation-derived flavors",null,"Beverages, candies, baked goods, snacks, processed foods",null,977038-73-5,182.2,2403,3,null
"D-8-P-MENTHENE-1,2-EPOXIDE",4,Safe,5,5,No,General_Approved,Flavoring Agent,FLAVORING AGENT OR ADJUVANT,"Natural plant extracts, essential oils, or fermentation-derived flavors",null,"Beverages, candies, baked goods, snacks, processed foods",null,203719-54-4,null,4655,24,2145
"DRAGON'S BLOOD, EXTRACT (DAEMONOROPS SPP. OR OTHER BOTANICAL SOURCES)",3,Safe,2,1,No,General_Approved,Flavoring Agent,FLAVORING AGENT OR ADJUVANT,"Natural plant extracts, essential oils, or fermentation-derived flavors",null,"Beverages, candies, baked goods, snacks, processed foods",null,9000-19-5,172.51,2404,3,null
DRIED ALGAE MEAL,4,Safe,5,5,No,General_Approved,Flavoring Agent,"FLAVOR ENHANCER, FLAVORING AGENT OR ADJUVANT","Natural plant extracts, essential oils, or fermentation-derived flavors",null,"Beverages, candies, baked goods, snacks, processed foods",null,977010-47-1,73.275,null,null,null
DULCIN--PROHIBITED,5,Moderate,8,9,No,General_Approved,Sweetener,NON-NUTRITIVE SWEETENER,"Stevia, Monk fruit extract, or Allulose",null,"Soft drinks, sugar-free products, baked goods, chewing gum, tabletop sweeteners",189.145,150-69-6,189.145,null,null,null
